# Reproducing Porniece et al. (2025) from dandiset 001474

Paper. Porniece M, Baker J, Ausfahl CD, Zhang SX, Andermann ML (2025). Obesity dysregulates
feeding-evoked response dynamics in hypothalamic satiety neurons. *Scientific Reports* 15, 35837.
[doi:10.1038/s41598-025-19800-2](https://doi.org/10.1038/s41598-025-19800-2). The claims file was
written from the open-access version of record (Europe PMC, PMC12521569), with the legends of
Supplementary Figures 1 to 3 (Supplementary Material 1).

Dandiset. [DANDI:001474](https://dandiarchive.org/dandiset/001474/0.251031.1725), version
0.251031.1725. The paper cites no DANDI version (its data statement says the data are in the
article and its supplements). This is the only published version, and the draft has the same 349
assets and size, so the published version is pinned.

What the dandiset holds. 349 NWB files, one per recording session, from 24 MC4R-Cre mice:
per-trial matrices (10 s before to 50 s after the tone, 50 Hz) of the Ensure pump signal, licks,
running and a Z-scored photometry signal computed by the depositors, the continuous raw 465 nm
trace, and notes with diet, fasting, consumed Ensure and body weights. It has no 405 nm control
trace, no trial table or tone time stamps, and no training sessions. Four files are second copies
of other files.

Readings. Each claim is tested under the readings its criterion lists. Z1 and Z2 are the claims
file's two Z-scorings, each with the per-mouse SD over all the mouse's recordings or over the
recordings of one diet phase (Z1p, Z2p). The depositors' Z-score (Z0) and dF/F0 are shown as
variants and do not decide outcomes. Nothing here is random, so no seed is needed.

The notebook reads all files in about 1.5 minutes and runs in about 2 minutes.

| Claim | Tested | Outcome |
|---|---|---|
| fig1k-early-late-recordings | yes | Partially reproduced |
| fig1k-lme | yes | Not reproduced |
| fig1l-early-late-mice | yes | Partially reproduced |
| supp1i-hfd-no-change-1-20s | yes | Reproduced |
| supp1h-dff-early-late | yes | Reproduced |
| fig1gh-hfd-less-ensure | yes | Reproduced |
| fig1c-hfd-heavier | yes | Partially reproduced |
| supp1cd-licking-declines | no | Blocked: methods underspecified |
| supp1e-hfd-less-locomotion | yes | Reproduced |
| fig2b-meal-size | yes | Reproduced |
| fig2c-delta-z-recordings | yes | Reproduced |
| fig2d-delta-z-mice | yes | Reproduced |
| fig2e-lick-vigor | yes | Partially reproduced |
| fig2fj-early-lick-by-meal-size | yes | Reproduced |
| fig2p-lick-rate-meal-size | yes | Reproduced |
| fig2km-elevated-early-at-matched-licking | yes | Reproduced |
| fig2q-midmeal-response-meal-size | yes | Reproduced |
| fig2r-weight-midmeal-response | yes | Reproduced |
| fig2s-weight-meal-length | yes | Reproduced |
| supp2a-volume-trials | yes | Reproduced |
| supp2h-delta-z-meal-size | yes | Not reproduced |
| supp2fg-locomotion-not-meal-size | yes | Reproduced |
| supp2j-variability-weight | yes | Reproduced |
| supp2k-sex-early-response | yes | Reproduced |
| supp2lp-fasting-no-effect | yes | Not reproduced |
| supp2q-hfd-more-training | no | Blocked: data missing |
| fig3h-early-response-diet-switch | yes | Partially reproduced |
| fig3i-ncd-increase-only | yes | Reproduced |
| fig3j-delta-z-diet-switch | yes | Not reproduced |
| fig3k-ensure-after-switch | yes | Reproduced |
| fig3-hfd-hfd-short-meals | yes | Not reproduced |
| fig3c-hfd-ncd-licking-recovers | yes | Reproduced |
| supp3c-ncd-hfd6w-dynamics-retained | yes | Reproduced |
| supp3j-body-weight-switch | yes | Reproduced |
| supp3m-weight-gain-response | yes | Partially reproduced (assumed) |
| supp3l-weight-not-meal-length | yes | Reproduced |
| supp3q-hfd-ncd-weight-loss-meal-length | yes | Reproduced (assumed) |

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import pvh

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)

started = time.time()
recs = pvh.load_all(workers=12)
print(f"loaded {len(recs)} files from dandiset {pvh.DANDISET_ID} version {pvh.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

In [ ]:
# Everything below uses `recs`, the per-file measures returned by pvh.load_all.
sigmas = pvh.add_z(recs)
sigmas_phase = pvh.add_z(recs, mouse_of=lambda r: (r["subject"], pvh.phase_of(r)), suffix="p")
T = pvh.session_groups(recs)
U = T[~T.duplicate].copy()
R = pvh.recording_measures(recs, U)
by_path = {r["path"]: r for r in recs}

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print("Files:", len(T), "; unique blobs:", T.blob.nunique())
dups = T[T.blob.duplicated(keep=False)].sort_values("blob")[["path", "identifier", "blob"]]
print("\nFiles that are byte-identical blobs of another file (deposited twice):")
print(dups.to_string(index=False))
print("\nThe 'ses-None' files and their identifiers:")
print(T[T.path.str.contains("ses-None")][["path", "identifier", "duplicate"]].to_string(index=False))
import hashlib
def digest(a):
    return hashlib.md5(np.ascontiguousarray(np.round(np.asarray(a, float), 6)).tobytes()).hexdigest()
content = pd.DataFrame([{"path": r["path"], "blob": r["blob"], "lick": digest(r["lick_1s"]),
                         "ensure": digest(r["ensure_onset"]),
                         "running": digest(r["run_signed"]["whole"]) if "run" in r else None,
                         "zscore": digest(r["z0"]["1-10"]) if "z0" in r else None,
                         "F0": digest(r["F0"]) if "F0" in r else None} for r in recs])
for col in ["lick", "running", "zscore", "F0"]:
    d = content[content[col].notna() & content[col].duplicated(keep=False)]
    print(f"Files sharing their {col} contents with another file: {len(d)}; all of them blob copies: "
          f"{bool(d.blob.duplicated(keep=False).all())}")
print("\nUnique recordings used below:", len(U))

Four of the six `ses-None` files are the same blob as a dated file of the same subject (same
identifier, same notes); each recording is counted once (the dated copy is kept). The other two
`ses-None` files (MPK122 2024-02-06 and MPK143 2024-03-14, by their identifiers) have no dated
copy and are kept. The lick, running, Z-score and F0 contents repeat only in the four blob
copies, so no other recording was deposited twice.

In [ ]:
info = U.groupby(["phase", "cohort", "subject"]).agg(
    sex=("sex", lambda s: "/".join(sorted(set(s)))), diet=("diet", "first"),
    switch=("switch", "first"), sessions=("path", "size"),
    trials=("n_trials", lambda s: sorted(set(s))), triggered_min=("n_triggered", "min"),
    triggered_max=("n_triggered", "max"), first=("date", "min"), last=("date", "max"))
print(info.to_string())
print("\nSex given per file; subjects whose files disagree:")
sx = T.groupby("subject").sex.unique()
print(sx[sx.map(len) > 1].to_string())
for s in ["MPK173", "MPK182"]:
    print(s, T[T.subject == s].groupby("phase").sex.value_counts().to_dict())

Cohort 1 (60-trial sessions, January to April 2024) holds 6 NCD and 5 HFD mice, as the Fig. 1
legend says. Cohort 2 (90-trial sessions, July to December 2024) holds 7 NCD mice but 6 HFD
mice, where the Fig. 2 legend says 7. The sex of MPK173 and MPK182 differs between their first
and second phase files. MPK173 has one 60-trial session in cohort 2; it is left out of the
90-trial analyses. The age is "P12W/" in every file, which cannot be right for recordings that
span a year (the diet switch happens at about 21 weeks, Fig. 3a).

In [ ]:
def sel_c1(min_trig=30):
    return R[(R.phase == "first") & (R.cohort == 1) & (R.n_triggered >= min_trig)]

def sel_c2(min_trig=30):
    return R[(R.phase == "first") & (R.cohort == 2) & (R.n_trials >= 89) & (R.n_triggered >= min_trig)]

def sel_switch(phase, min_trig=30):
    return R[R.switch.notna() & (R.phase == phase) & (R.n_triggered >= min_trig)]

def count(df, by):
    return df.groupby(by).agg(recordings=("path", "size"), mice=("subject", "nunique"))

print("Cohort 1, at least 30 triggered trials (Fig. 1: NCD 39/6, HFD 33/5):")
print(count(sel_c1(), "diet").to_string())
print("  all cohort 1 recordings:", count(sel_c1(0), "diet").to_dict())
print("\nCohort 2, 90-trial sessions, at least 30 triggered (Fig. 2a-e: NCD 59/7, HFD 54/7):")
print(count(sel_c2(), "diet").to_string())
print("  at least 15 triggered (meal-size groups sum to NCD 57, HFD 53):", count(sel_c2(15), "diet").to_dict())
print("  all 90-trial sessions:", count(sel_c2(0), "diet").to_dict())
print("\nDiet switch, at least 30 triggered (legend, first/second: NCD->NCD 44/43, NCD->HFD6w 47/36, "
      "HFD->NCD 52/42, HFD->HFD 35/23; mice 6, 6, 7, 4):")
print(pd.concat({"first": count(sel_switch("first"), "switch"),
                 "second": count(sel_switch("second"), "switch")}, axis=1).to_string())
print("\nTrials per session in the diet-switch phases:")
print(R[R.switch.notna()].groupby(["switch", "phase"]).n_trials.value_counts().unstack(fill_value=0).to_string())

The groups of the diet switch are the same mice as cohorts 1 and 2: the first-diet recordings
are their cohort recordings, and the second-diet recordings are separate files whose notes give
`startdiet` and `2nd_diet`. The first-diet counts match the Fig. 3 legend for NCD->NCD (44) and
fall short by 1 to 5 for the others; the second-diet counts match it for NCD->NCD (43),
NCD->HFD6w (36) and HFD->HFD (23), and are one short for HFD->NCD (41 against 42). The diet-switch
groups mix 60-trial sessions (cohort 1 mice) with 90-trial sessions (cohort 2 mice), where the
claims file describes 90-trial sessions.

In [ ]:
rates = pd.DataFrame([{**{f"rate_{k}": v for k, v in r["rates"].items()}} for r in recs])
print("starting_time rate attribute (Hz) per series, as stored:")
for c in rates:
    print(" ", c, rates[c].value_counts(dropna=False).to_dict())
print("\nUnits as stored:", recs[0]["units"])
print("Samples per trial matrix:", sorted({r["n_samples"] for r in recs}))
lens = pd.Series([r["raw_len"] / pvh.FS / 60 for r in recs if "raw_len" in r])
print("Raw 465 trace length at 50 Hz (min):", lens.groupby(pd.Series([r["n_trials"] for r in recs if "raw_len" in r])).agg(["min", "median", "max"]).round(1).to_dict("index"))
print("Raw trace power above 10.5 Hz, fraction of total: max", f"{max(r['raw_power_above10'] for r in recs if 'raw_len' in r):.2e}")
print("Files whose raw trace starts with NaN samples:", sum(r.get("raw_nan", 0) > 0 for r in recs), "; NaN samples:", sorted({r.get('raw_nan', 0) for r in recs}))
print("\nSeries missing from some files:")
allser = {"ensure_signal", "fiber_465_trace", "fiber_signal_zscore", "lick_signal", "running_signal"}
miss = pd.Series([",".join(sorted(allser - set(r["series"]))) or "-" for r in recs]).value_counts()
print(miss.to_string())
cc = pd.DataFrame([dict(path=r["path"], **pvh.column_counts(r)) for r in recs])
mism = cc[cc.drop(columns="path").nunique(axis=1) > 1]
print("\nFiles whose trial matrices differ in number of columns:", len(mism))
print(mism.drop(columns="path").value_counts().to_string())

The trial matrices have 3001 samples, 60 s at 50 Hz, as the Methods' 60-s trials; the
`starting_time` rate is stored as 1 Hz in about half the files and 50 Hz in the others, so the
stored rate is wrong in about half the files (the raw trace lengths, 67 to 106 min, fit 50 Hz and
the session lengths of the Methods). The raw trace holds no power above 10 Hz: it is the
low-passed trace of the Methods. In 27 files one series has more trial columns than the Ensure
matrix (one more in most, 90 against 60 or 46 in two); trials beyond the Ensure matrix are
dropped. Running is missing from 9 files, the deposited Z-score from 27 and the raw trace from 6.

In [ ]:
on = np.concatenate([r["ensure_onset"][r["triggered"]] for r in recs])
print("Pump signal range:", (min(r["ensure_range"][0] for r in recs), max(r["ensure_range"][1] for r in recs)))
print("Sample of pump onset in triggered trials (tone at sample 500):", pd.Series(on).value_counts().to_dict())
lick_vals = sorted({v for r in recs for v in r["lick_values"]})
print("Lick matrix values:", lick_vals)
cue = np.concatenate([np.asarray(r["lick"]["cue"])[:len(r["triggered"])] for r in recs])
trg = np.concatenate([r["triggered"][:len(r["lick"]["cue"])] for r in recs])
print("Cue-window (0-1 s) lick rate, triggered trials: mean", cue[trg].mean().round(2),
      "; untriggered trials: mean", cue[~trg].mean().round(2), "; untriggered trials with no cue lick:",
      f"{(cue[~trg] == 0).mean():.3f}")
pre = np.concatenate([np.asarray(r["lick"]["pre"])[:len(r["triggered"])] for r in recs])
print("Pre-tone (-10-0 s) lick rate: triggered", pre[trg].mean().round(3), "untriggered", pre[~trg].mean().round(3))
print("Running speed range (cm/s):", (min(r["run_range"][0] for r in recs if "run" in r), max(r["run_range"][1] for r in recs if "run" in r)))
print("\nNotes values: Ensure (mL?)", U.ensure.describe().round(2).to_dict())
print("Body weight (g)", U.bw.describe().round(2).to_dict())
print("Conditions:", U.condition.value_counts().to_dict())

The pump signal rises at sample 549 to 552 in every triggered trial, 1 s after the tone, as the
Methods' delivery at t = 1 s, so sample 500 is the tone. Untriggered trials have a cue-window
lick rate near zero and triggered trials a high one, which shows that the lick matrix shares the
clock of the trial structure: a lick in the 1-s cue window triggers delivery (Methods). The
Ensure value in the notes is read as mL (range 0 to about 4; 50 uL per triggered trial gives up
to 4.5 mL in 90 trials), which is consistent with the data, not shown by them. The notes carry
no unit for body weight; the values fit grams.

In [ ]:
al = pd.DataFrame([dict(path=r["path"], n_trials=r["n_trials"],
                        offset=r.get("offset_corr", r.get("offset")),
                        median_r=np.median(r["align_r"]), match=r["z_raw_match"])
                   for r in recs if "align_r" in r])
print("Files with both raw trace and deposited Z-score:", len(al))
print("Deposited Z-score columns matched to the raw trace (median r > 0.5, offset within 25 samples of 14771):",
      int(al.match.sum()), "; not matched:", int((~al.match).sum()))
print("Offsets of matched files (samples, 50 Hz):", al[al.match].offset.value_counts().head(8).to_dict())
print("Median correlation, matched files:", al[al.match].median_r.describe().round(3).to_dict())
print("\nFiles whose deposited Z-score does not match their raw trace:")
print(al[~al.match].sort_values("median_r").round(3).to_string(index=False))
print("\nOffset source for the raw measures:", U.offset_source.value_counts(dropna=False).to_dict())
# Does the deposited Z-score respond like the raw trace? per-recording correlation of early-late values
x = R[R["z0|1-10|D15"].notna() & R["z1|1-10|D15"].notna()]
x = x.merge(al[["path", "match"]], on="path", how="left")
for m, g in x.groupby("match"):
    print(f"match={m}: r(D15 deposited Z, D15 of Z1 from raw) over recordings = {stats.pearsonr(g['z0|1-10|D15'], g['z1|1-10|D15'])[0]:.3f} (n={len(g)})")

The raw trace has no tone time stamps. In the files where the deposited Z-score matches the raw
trace, the first trial starts at sample 14,750 to 14,773, that is 295 s, so the first tone is 5 min
5 s into the recording (the Methods' 5-min baseline), and every later trial is 3000 samples (60 s)
after the previous one, within 3 samples. This is the shared clock of the raw trace and the trial
matrices. In 32 files the deposited Z-score does not match the raw trace of the same file at any
offset (median correlation below 0.5): the deposited Z-score and the raw trace of these files
cannot both come from the same recording, or the Z-score was computed from a differently
processed signal. Where that happens, and in the files without a Z-score, the raw measures use
the modal offset (an assumption that the session timing is the same). The deposited Z-score is
noisier from sample to sample than the raw trace (it was not computed from it by a linear
transform; checked while preparing), so it is used only as a variant (Z0).

In [ ]:
sg = pd.DataFrame([(m, s1, s2) for m, (s1, s2) in sigmas.items()], columns=["mouse", "sigma_Z1 (raw units)", "sigma_Z2 (dF/F0)"])
print(sg.round(4).to_string(index=False))
lost = [(r["path"], int(r["lost"].sum())) for r in recs if "lost" in r and r["lost"].any()]
print("\nTrials left out of the raw measures because the signal drops below 20% of the recording's median F0:",
      sum(n for _, n in lost), lost)
print("\nPer-trial response ranges over all triggered trials, 1-10 s window:")
for z in ["z1", "z2", "z1p", "z2p", "z0", "dff"]:
    v = np.concatenate([pvh._tv(r[z]["1-10"], r["triggered"]) for r in recs if z in r])
    print(f"  {z}: min {np.nanmin(v):.2f}, median {np.nanmedian(v):.3f}, max {np.nanmax(v):.2f}")

## Fig. 1: cohort 1

Readings used throughout (claims file, opening comment): response windows W 0-10, 1-10, 1-9 s;
Z1 and Z2 as defined there, each with the per-mouse sigma taken over all the mouse's recordings
(z1, z2) or over its recordings of the same diet phase (z1p, z2p), two readings of the Methods'
"a single Z-score per animal" (see `pvh`); early = first 15 triggered trials, late = last 15;
D5 and D15 for the delta Z-score. Z0, the depositors' Z-score, and dF/F0 are shown as variants
and do not decide outcomes. For cohort 1 and 2 claims Z1p and Z2p use only the first-diet
recordings.

In [ ]:
ZS, ZV = ["z1", "z2", "z1p", "z2p"], ["z0"]
W1 = ["0-10", "1-10", "1-9"]

def means(df, z, w, phases=("early", "late"), by="diet"):
    cols = [f"{z}|{w}|{p}" for p in phases]
    g = df.groupby(by)[cols]
    out = pd.concat({"mean": g.mean(), "sem": g.sem(), "n": g.count()}, axis=1)
    out.columns = [f"{a}_{b.split('|')[-1]}" for a, b in out.columns]
    return out

def mouse_table(df, cols, by="diet"):
    return df.groupby([by, "subject"])[cols].mean().reset_index()

### fig1k-early-late-recordings

> Per recording, PVH MC4R consumption responses in NCD mice are small early in the meal (Z 0.12
> +- 0.11) and larger late (0.86 +- 0.09); in HFD mice they are already high early (0.72 +-
> 0.09) and do not increase late (0.79 +- 0.09).

In [ ]:
EXP_1K = {("NCD", "early"): (0.12, 0.11), ("NCD", "late"): (0.86, 0.09),
          ("HFD", "early"): (0.72, 0.09), ("HFD", "late"): (0.79, 0.09)}
c1 = sel_c1()
rows = []
for z in ZS + ZV + ["dff"]:
    for w in W1:
        m = means(c1, z, w)
        row = {"Z": z, "W": w}
        inside = []
        for (d, p), (v, tol) in EXP_1K.items():
            row[f"{d} {p}"] = m.loc[d, f"mean_{p}"]
            row[f"{d} {p} sem"] = m.loc[d, f"sem_{p}"]
            inside.append(abs(m.loc[d, f"mean_{p}"] - v) <= tol)
        row["n in tolerance"] = sum(inside)
        row["all four"] = all(inside)
        row["NCD late > NCD early"] = row["NCD late"] > row["NCD early"]
        row["HFD early > NCD early"] = row["HFD early"] > row["NCD early"]
        rows.append(row)
F1K = pd.DataFrame(rows)
print("Cohort 1, recordings with >= 30 triggered trials:", count(c1, "diet").to_dict())
print(F1K.round(3).to_string(index=False))
print("\nDecides the outcome (Z1, Z2): all four in tolerance under some combination:",
      bool(F1K[F1K.Z.isin(ZS)]["all four"].any()),
      "; partial condition under some combination:",
      bool((F1K[F1K.Z.isin(ZS)]["NCD late > NCD early"] & F1K[F1K.Z.isin(ZS)]["HFD early > NCD early"]).any()))

In [ ]:
print("Per mouse breakdown (Z2, 1-10 s), recording means:")
print(c1.groupby(["diet", "subject"])[["z2|1-10|early", "z2|1-10|late"]].agg(["mean", "count"]).round(3).to_string())
fig, ax = plt.subplots(1, 2, figsize=(8, 3.2), sharey=True)
for a, d in zip(ax, ["NCD", "HFD"]):
    g = c1[c1.diet == d]
    for _, r in g.iterrows():
        a.plot([0, 1], [r["z2|1-10|early"], r["z2|1-10|late"]], color="0.7", lw=0.7)
    a.errorbar([0, 1], [g["z2|1-10|early"].mean(), g["z2|1-10|late"].mean()],
               yerr=[g["z2|1-10|early"].sem(), g["z2|1-10|late"].sem()], color="k", marker="o")
    a.set_xticks([0, 1], ["early", "late"]); a.set_title(f"{d} (Z2, 1-10 s)")
ax[0].set_ylabel("response (Z2)")
lo, hi = c1[["z2|1-10|early", "z2|1-10|late"]].min().min(), c1[["z2|1-10|early", "z2|1-10|late"]].max().max()
print(f"Plotted range: {lo:.2f} to {hi:.2f}; all points inside the axes.")
plt.tight_layout(); plt.show()

**Result: partially reproduced.** No combination of window and Z reading puts all four means
within tolerance; the best put two in (Z2 and Z2p: HFD early 0.72 to 0.80 and HFD late 0.73 to
0.82 against 0.72 and 0.79, while NCD early is 0.33 to 0.43 against 0.12 +- 0.11 and NCD late 1.01
to 1.22 against 0.86 +- 0.09). Under every combination NCD late is above NCD early and HFD early is
above NCD early, which is the partial condition. Under Z1 and Z1p every value is smaller (NCD
early 0.10 to 0.13, late 0.38 to 0.43). The depositors' Z-score (Z0, a variant) gives NCD early
-0.17 to -0.23, also outside tolerance.

### fig1k-lme

> In a linear mixed-effects model of recording-level responses with mouse as a random
> intercept, HFD early-meal responses exceed NCD early-meal responses and the diet by meal-phase
> interaction is significant.

In [ ]:
def lme_diet_phase(df, z, w, group="diet", ref="NCD"):
    long = pd.concat([
        pd.DataFrame({"Value": df[f"{z}|{w}|early"], "Diet": df[group], "Phase": "Early", "Mouse": df.subject}),
        pd.DataFrame({"Value": df[f"{z}|{w}|late"], "Diet": df[group], "Phase": "Late", "Mouse": df.subject})])
    long = long.dropna()
    m = smf.mixedlm(f"Value ~ C(Diet, Treatment('{ref}')) * C(Phase, Treatment('Early'))", long,
                    groups=long["Mouse"]).fit(reml=True)
    return m

rows = []
for z in ZS + ZV:
    for w in W1:
        m = lme_diet_phase(c1, z, w)
        names = m.params.index
        d = [n for n in names if "Diet" in n and ":" not in n][0]
        i = [n for n in names if ":" in n][0]
        rows.append({"Z": z, "W": w, "HFD coef": m.params[d], "p HFD": m.pvalues[d],
                     "interaction": m.params[i], "p interaction": m.pvalues[i],
                     "converged": m.converged})
LME1 = pd.DataFrame(rows)
LME1["full (p<0.001 both, signs)"] = (LME1["HFD coef"] > 0) & (LME1["p HFD"] < 0.001) & (LME1.interaction < 0) & (LME1["p interaction"] < 0.001)
LME1["partial (p<0.05 both, signs)"] = (LME1["HFD coef"] > 0) & (LME1["p HFD"] < 0.05) & (LME1.interaction < 0) & (LME1["p interaction"] < 0.05)
print(f"Rows: {2 * len(c1)} (two per recording), mice: {c1.subject.nunique()}")
print(LME1.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: not reproduced.** The diet by meal-phase interaction is negative with p < 0.001 under
every Z1, Z2, Z1p and Z2p combination (p 8e-07 to 1.2e-05), but the HFD coefficient (HFD against
NCD early in the meal) is positive with p 0.09 to 0.35, so neither the full nor the partial
condition holds. With the depositors' Z-score (Z0, a variant that does not decide the outcome) both
hold at p < 0.001 (HFD coefficient 0.80 to 0.88, p 1.3e-06 to 4.7e-06; interaction -0.58 to
-0.64, p 4e-05 to 6e-05).

### fig1l-early-late-mice

> Per mouse, every NCD mouse increases its response from early to late in the meal (Z 0.19 +-
> 0.24 to 0.91 +- 0.13), while HFD mice do not change (0.73 +- 0.19 to 0.80 +- 0.20).

In [ ]:
EXP_1L = {("NCD", "early"): (0.19, 0.24), ("NCD", "late"): (0.91, 0.13),
          ("HFD", "early"): (0.73, 0.19), ("HFD", "late"): (0.80, 0.20)}
rows = []
for z in ZS + ZV:
    for w in W1:
        mt = mouse_table(c1, [f"{z}|{w}|early", f"{z}|{w}|late"])
        row = {"Z": z, "W": w}
        inside = []
        for (d, p), (v, tol) in EXP_1L.items():
            x = mt[mt.diet == d][f"{z}|{w}|{p}"]
            row[f"{d} {p}"] = x.mean(); row[f"{d} {p} sem"] = x.sem()
            inside.append(abs(x.mean() - v) <= tol)
        n = mt[mt.diet == "NCD"]
        row["NCD mice late>early"] = f"{int((n[f'{z}|{w}|late'] > n[f'{z}|{w}|early']).sum())}/{len(n)}"
        for d in ["NCD", "HFD"]:
            x = mt[mt.diet == d]
            t = stats.ttest_rel(x[f"{z}|{w}|late"], x[f"{z}|{w}|early"])
            row[f"p {d}"] = t.pvalue
            row[f"{d} late-early"] = (x[f"{z}|{w}|late"] - x[f"{z}|{w}|early"]).mean()
        row["means in tol"] = sum(inside)
        row["tests as claimed"] = (row["NCD mice late>early"] == f"{len(n)}/{len(n)}") and row["p NCD"] < 0.05 and row["p HFD"] >= 0.05
        row["full"] = row["tests as claimed"] and all(inside)
        rows.append(row)
F1L = pd.DataFrame(rows)
print(F1L.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nPer mouse values (Z2, 1-10 s):")
print(mouse_table(c1, ["z2|1-10|early", "z2|1-10|late"]).round(3).to_string(index=False))

**Result: partially reproduced.** Under every Z1, Z2, Z1p and Z2p combination all 6 NCD mice
have late above early, the NCD paired t-test has p 0.0013 to 0.035 and the HFD one p 0.11 to 0.96,
as the claim states; but no combination puts all four means within tolerance (at most three:
under Z2 and Z2p NCD late is 1.04 to 1.25 against 0.91 +- 0.13). Under Z0 one NCD mouse of six
does not increase.

### supp1i-hfd-no-change-1-20s

> With a longer response window of 1 to 20 s after the cue, HFD mice still show no change in
> response from early to late in the meal.

In [ ]:
def tukey(values, labels):
    """Tukey HSD over all groups; a table of every pair with unrounded p-values and
    diff = mean(group1) - mean(group2)."""
    import itertools
    tk = pairwise_tukeyhsd(np.asarray(values, float), np.asarray(labels).astype(str))
    pairs = list(itertools.combinations(tk.groupsunique, 2))
    # statsmodels' meandiffs are mean(group2) - mean(group1)
    return pd.DataFrame({"group1": [a for a, _ in pairs], "group2": [b for _, b in pairs],
                         "diff": -np.asarray(tk.meandiffs), "p-adj": tk.pvalues})

def anova_tukey(long, value="Value", f1="A", f2="B"):
    import statsmodels.api as sm
    model = smf.ols(f"{value} ~ C({f1}) * C({f2})", long).fit()
    table = sm.stats.anova_lm(model, typ=2)
    tkd = tukey(long[value], long[f1].astype(str) + "|" + long[f2].astype(str))
    return table, tkd

def tukey_p(tkd, a, b):
    """Tukey p and mean(a) - mean(b)."""
    r = tkd[((tkd.group1 == a) & (tkd.group2 == b)) | ((tkd.group1 == b) & (tkd.group2 == a))].iloc[0]
    return float(r["p-adj"]), float(r["diff"]) * (1 if r.group1 == a else -1)

rows = []
for z in ZS + ZV:
    for w in ["0-20", "1-20"]:
        mt = mouse_table(c1, [f"{z}|{w}|early", f"{z}|{w}|late"])
        long = pd.concat([pd.DataFrame({"Value": mt[f"{z}|{w}|{p}"], "A": mt.diet, "B": p}) for p in ["early", "late"]])
        tab, tkd = anova_tukey(long)
        ph, dh = tukey_p(tkd, "HFD|late", "HFD|early")
        pn, dn = tukey_p(tkd, "NCD|late", "NCD|early")
        rows.append({"Z": z, "W": w, "HFD early": mt[mt.diet == "HFD"][f"{z}|{w}|early"].mean(),
                     "HFD late": mt[mt.diet == "HFD"][f"{z}|{w}|late"].mean(),
                     "HFD late-early": dh, "Tukey p HFD": ph, "NCD late-early": dn, "Tukey p NCD": pn,
                     "p interaction (ANOVA)": tab.loc["C(A):C(B)", "PR(>F)"]})
S1I = pd.DataFrame(rows)
print("Mice:", count(c1, "diet").mice.to_dict())
print(S1I.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** With the 0-20 and 1-20 s windows, the Tukey comparison of HFD early
against HFD late has p 0.88 to 0.997 under every Z reading (late minus early -0.02 to 0.08). The
NCD early-late comparison is not significant either (p 0.07 to 0.64) under Z1, Z2, Z1p and Z2p; it
is under Z0 (p 0.03 and 0.04).

### supp1h-dff-early-late

> Expressed as dF/F, HFD mice have stronger consumption responses than NCD mice at the start of
> the meal, and these persist to the end of the feeding bout.

In [ ]:
rows = []
for w in W1:
    m = means(c1, "dff", w)
    he, hl, ne = m.loc["HFD", "mean_early"], m.loc["HFD", "mean_late"], m.loc["NCD", "mean_early"]
    rows.append({"W": w, "NCD early": ne, "NCD late": m.loc["NCD", "mean_late"], "HFD early": he,
                 "HFD early sem": m.loc["HFD", "sem_early"], "HFD late": hl,
                 "HFD early > NCD early": he > ne, "HFD late >= HFD early - sem": hl >= he - m.loc["HFD", "sem_early"],
                 "effect HFD-NCD early": he - ne})
S1H = pd.DataFrame(rows)
print(S1H.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nPer mouse (1-10 s, dF/F0):")
print(c1.groupby(["diet", "subject"])[["dff|1-10|early", "dff|1-10|late"]].mean().round(4).to_string())

**Result: reproduced.** In every window the HFD early-meal dF/F0 (0.025 to 0.027) is above the
NCD one (0.010 to 0.013), and the HFD late-meal value (0.026 to 0.027) is at least the HFD early
value minus its s.e.m.

### fig1gh-hfd-less-ensure

> Despite calorie matching, HFD mice consume less Ensure than NCD mice, per recording session and
> per mouse.

In [ ]:
rows = []
for label, df in [(">= 30 triggered", sel_c1()), ("all sessions", sel_c1(0))]:
    for vol, col in [("recorded volume (notes)", "ensure"), ("triggered x 50 uL", "n_triggered")]:
        x = df.assign(v=df[col] * (0.05 if col == "n_triggered" else 1))
        g = stats.ttest_ind(x[x.diet == "HFD"].v, x[x.diet == "NCD"].v)
        mm = x.groupby(["diet", "subject"]).v.mean().reset_index()
        h = stats.ttest_ind(mm[mm.diet == "HFD"].v, mm[mm.diet == "NCD"].v)
        rows.append({"sessions": label, "volume": vol,
                     "NCD sessions mean": x[x.diet == "NCD"].v.mean(), "HFD sessions mean": x[x.diet == "HFD"].v.mean(),
                     "n": f"{(x.diet == 'NCD').sum()}/{(x.diet == 'HFD').sum()}", "t g": g.statistic, "p g": g.pvalue,
                     "NCD mice mean": mm[mm.diet == "NCD"].v.mean(), "HFD mice mean": mm[mm.diet == "HFD"].v.mean(),
                     "t h": h.statistic, "p h": h.pvalue})
F1GH = pd.DataFrame(rows)
print("Recordings with no recorded volume:", int(sel_c1(0).ensure.isna().sum()))
print(F1GH.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nPer mouse recorded volume (mL), all sessions:")
print(sel_c1(0).groupby(["diet", "subject"]).ensure.agg(["mean", "count"]).round(2).to_string())

**Result: reproduced.** With the recorded volume, HFD sessions consume 1.62 mL against 2.63 mL
for NCD (p 5.7e-09) and HFD mice 1.61 against 2.62 mL (p 0.0019) in the recordings with at least 30
triggered trials; with all sessions p is 1.7e-10 and 0.0017. The triggered trials times 50 uL
give the same result.

### fig1c-hfd-heavier

> During food restriction, HFD mice are about 30% heavier than NCD mice.

In [ ]:
rows = []
for label, df in [("all recording days", sel_c1(0)), (">= 30 triggered", sel_c1())]:
    mm = df.groupby(["diet", "subject"]).bw.mean().reset_index()
    a = mm[mm.diet == "HFD"].bw.mean() / mm[mm.diet == "NCD"].bw.mean() - 1
    b = df[df.diet == "HFD"].bw.mean() / df[df.diet == "NCD"].bw.mean() - 1
    rows.append({"days": label, "(a) mouse means": a, "(b) pooled": b,
                 "NCD mean g (mice)": mm[mm.diet == "NCD"].bw.mean(), "HFD mean g (mice)": mm[mm.diet == "HFD"].bw.mean()})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("Expected 0.30 +- 0.03; partial range 0.20 to 0.40.")
print("\nPer mouse body weight on recording days (g):")
print(sel_c1(0).groupby(["diet", "subject"]).bw.agg(["mean", "min", "max", "count"]).round(2).to_string())
print("\nAd lib body weight in the notes (g), per mouse:", sel_c1(0).groupby(["diet", "subject"]).adlib_bw.agg(lambda s: sorted(set(s))).to_dict())

**Result: partially reproduced.** HFD mice are 35.2% heavier than NCD mice by mouse means
(27.3 against 20.2 g) and 34.4% by pooled recording days, outside 30 +- 3% and inside 20 to 40%.

### supp1cd-licking-declines

> Both NCD and HFD mice reduce their licking from the early to the late phase of the session.

In [ ]:
def lick_auc(df, norm, win):
    """Per recording, the early and late mean lick-rate curves (1-s bins, -10 to 50 s) over
    triggered trials; per mouse the mean over its recordings; AUC over the window of the curve
    normalized to the mean rate of the 10 s before the tone, by subtraction or division."""
    out = []
    for _, row in df.iterrows():
        r = by_path[row.path]
        L = r["lick_1s"][:, :len(r["triggered"])][:, r["triggered"][:r["lick_1s"].shape[1]]]
        for p, cols in [("early", L[:, :15]), ("late", L[:, -15:])]:
            out.append({"subject": row.subject, "diet": row.diet, "phase": p, **{f"b{i}": v for i, v in enumerate(cols.mean(axis=1))}})
    c = pd.DataFrame(out).groupby(["diet", "subject", "phase"]).mean().reset_index()
    B = c[[f"b{i}" for i in range(60)]].to_numpy()
    base = B[:, :10].mean(axis=1, keepdims=True)
    lo, hi = {"1-10": (11, 20), "0-50": (10, 60)}[win]
    seg = B[:, lo:hi]
    c["auc"] = (seg - base).sum(axis=1) if norm == "subtract" else (seg / base).sum(axis=1)
    c["base"] = base[:, 0]
    return c[["diet", "subject", "phase", "auc", "base"]]

rows = []
for norm in ["subtract", "divide"]:
    for win in ["1-10", "0-50"]:
        c = lick_auc(c1, norm, win)
        for d in ["NCD", "HFD"]:
            x = c[c.diet == d].pivot(index="subject", columns="phase", values="auc")
            for test in ["unpaired", "paired"]:
                t = stats.ttest_ind(x.late, x.early) if test == "unpaired" else stats.ttest_rel(x.late, x.early)
                rows.append({"norm": norm, "AUC window": win, "diet": d, "test": test,
                             "early": x.early.mean(), "late": x.late.mean(), "late-early": (x.late - x.early).mean(),
                             "t": t.statistic, "p": t.pvalue, "finite": bool(np.isfinite(x.to_numpy()).all()),
                             "late<early, p<0.05": (x.late.mean() < x.early.mean()) and t.pvalue < 0.05})
S1D = pd.DataFrame(rows)
print(S1D.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nPre-tone lick rate (licks/s) per mouse and phase (the divisor of the 'divide' readings):")
print(lick_auc(c1, "subtract", "1-10").pivot_table(index=["diet", "subject"], columns="phase", values="base").round(3).to_string())
print("\nUnnormalized mean cue lick rate (0-1 s), mouse means:")
print(mouse_table(c1, ["lick_cue|early", "lick_cue|late"]).groupby("diet")[["lick_cue|early", "lick_cue|late"]].mean().round(2).to_string())

**Result: blocked (methods underspecified).** The readings disagree. With the AUC normalized by
subtracting the pre-tone lick rate, late is below early with p < 0.05 in both diets under both
windows and both tests (p 0.0001 to 0.031). With division by the pre-tone rate, late is above
early in both diets (p 0.31 to 0.78), because the pre-tone rate falls from early to late (for
example MPK134 0.132 to 0.037 licks/s). The criterion blocks the claim when the readings
disagree.

### supp1e-hfd-less-locomotion

> Food-seeking locomotion is negligible in HFD mice compared with NCD mice, and falls further
> near the end of the meal.

In [ ]:
rows = []
print("Fraction of running samples below zero, cohort 1 files:",
      round(float(np.mean([by_path[p]["run_neg_fraction"] for p in c1.path if "run_neg_fraction" in by_path[p]])), 3))
print("Speed is the absolute value of the signed running signal; the signed mean is shown as a variant.")
for k, label in [("whole", "(a) whole trial -10 to 50 s"), ("pre", "(b) 10 s before tone"),
                 ("signed:whole", "variant: signed, whole trial"), ("signed:pre", "variant: signed, before tone")]:
    kk = k.split(":")[-1]
    pref = "runsigned" if k.startswith("signed") else "run"
    c1 = c1.assign(**{f"run_{k}|early": c1[f"{pref}_{kk}|early"], f"run_{k}|late": c1[f"{pref}_{kk}|late"]})
    mt = mouse_table(c1, [f"run_{k}|early", f"run_{k}|late"])
    g = mt.groupby("diet")[[f"run_{k}|early", f"run_{k}|late"]].agg(["mean", "sem"])
    ne, nl = g.loc["NCD", (f"run_{k}|early", "mean")], g.loc["NCD", (f"run_{k}|late", "mean")]
    he, hl = g.loc["HFD", (f"run_{k}|early", "mean")], g.loc["HFD", (f"run_{k}|late", "mean")]
    rows.append({"window": label, "NCD early": ne, "NCD late": nl, "HFD early": he, "HFD late": hl,
                 "HFD<NCD both phases": (he < ne) and (hl < nl), "HFD late<HFD early": hl < he})
print("Recordings without running data in cohort 1 (>= 30 triggered):", int(c1["run_whole|all"].isna().sum()))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("\nPer mouse (cm/s):")
print(mouse_table(c1, ["run_whole|early", "run_whole|late", "run_pre|early", "run_pre|late"]).round(3).to_string(index=False))
c1 = sel_c1()

**Result: reproduced.** Under both windows, HFD mean speed over mice is below the NCD mean in
both phases (whole trial: HFD 0.71 and 0.44 against NCD 3.97 and 1.29 cm/s) and HFD late is below
HFD early (0.44 against 0.71; before the tone 0.53 against 0.84). The signed means give the same
result.

## Fig. 2 and Supp. Fig. 2: cohort 2

90-trial sessions of cohort 2 (files with 89 trial columns are counted as 90-trial sessions
missing the last trial). Windows W 1-9 and 1-10 s; Z1, Z2 as above; Z0 a variant.

In [ ]:
W2 = ["1-9", "1-10"]
c2 = sel_c2()
c2_15 = sel_c2(15)
BINS = [(15, 29), (30, 44), (45, 59), (60, 74), (75, 90)]
def meal_bin(n):
    for a, b in BINS:
        if a <= n <= b:
            return f"{a}-{b}"
    return None
R["meal_bin"] = R.n_triggered.map(meal_bin)
c2 = sel_c2(); c2_15 = sel_c2(15)
print("Meal-size groups, cohort 2 (legend: NCD 0, 6, 10, 13, 28; HFD 6, 13, 23, 7, 4):")
print(c2_15.groupby(["diet", "meal_bin"]).size().unstack(fill_value=0).to_string())

def pearson(x, y):
    ok = np.isfinite(x) & np.isfinite(y)
    r, p = stats.pearsonr(x[ok], y[ok])
    return r, p, int(ok.sum())

### fig2b-meal-size

> In 90-trial sessions NCD mice consume Ensure in 70.1 +- 2.1 trials and HFD mice in 52.9 +-
> 1.98 trials.

In [ ]:
rows = []
for label, k in [("(a) all 90-trial", 0), ("(b) >= 30 triggered", 30), ("(c) >= 15 triggered", 15)]:
    x = sel_c2(k)
    n, h = x[x.diet == "NCD"].n_triggered, x[x.diet == "HFD"].n_triggered
    t = stats.ttest_ind(n, h)
    rows.append({"sessions": label, "NCD mean": n.mean(), "NCD sem": n.sem(), "n NCD": len(n),
                 "HFD mean": h.mean(), "HFD sem": h.sem(), "n HFD": len(h), "t": t.statistic, "p": t.pvalue,
                 "NCD in tol": abs(n.mean() - 70.1) <= 2.1, "HFD in tol": abs(h.mean() - 52.9) <= 1.98})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nPer mouse mean triggered trials (all 90-trial sessions):")
print(sel_c2(0).groupby(["diet", "subject"]).n_triggered.agg(["mean", "min", "max", "count"]).round(1).to_string())

**Result: reproduced.** In recordings with at least 30 triggered trials, NCD sessions trigger
70.1 +- 2.1 and HFD sessions 53.0 +- 2.0 trials (Student's t-test p 1.0e-07), both within
tolerance; with all 90-trial sessions HFD is 51.3, outside tolerance.

### fig2c-delta-z-recordings

> Per recording, the change in response from early to late in the meal is 0.52 +- 0.09 in NCD
> mice and only 0.12 +- 0.08 in HFD mice.

In [ ]:
rows = []
for z in ZS + ZV:
    for w in W2:
        for D in ["D5", "D15"]:
            col = f"{z}|{w}|{D}"
            n, h = c2[c2.diet == "NCD"][col].dropna(), c2[c2.diet == "HFD"][col].dropna()
            t = stats.ttest_ind(n, h)
            rows.append({"Z": z, "W": w, "D": D, "NCD": n.mean(), "NCD sem": n.sem(), "HFD": h.mean(), "HFD sem": h.sem(),
                         "n": f"{len(n)}/{len(h)}", "t": t.statistic, "p": t.pvalue,
                         "in tol": f"{int(abs(n.mean() - 0.52) <= 0.09)}{int(abs(h.mean() - 0.12) <= 0.08)}",
                         "NCD>HFD p<0.05": (n.mean() > h.mean()) and t.pvalue < 0.05})
F2C = pd.DataFrame(rows)
print(F2C.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nPer mouse (Z2, 1-9 s, D5): mean and count over recordings")
print(c2.groupby(["diet", "subject"])["z2|1-9|D5"].agg(["mean", "count"]).round(3).to_string())

**Result: reproduced.** Under Z2 (1-9 s, D15) the delta Z-score is 0.61 +- 0.07 for NCD and
0.20 +- 0.06 for HFD recordings, both within tolerance, with NCD above HFD (p 5.4e-05); Z2p (1-9
and 1-10 s, D15) also puts both in. Under the Methods' D5 the HFD value is negative (-0.07 to
-0.02) and out of tolerance; under Z1 and Z1p both values are out (NCD 0.10 to 0.17). Every
combination gives NCD above HFD with p < 0.001.

### fig2d-delta-z-mice

> Per mouse, the early-to-late change in response is 0.56 +- 0.10 in NCD mice and 0.28 +- 0.09 in
> HFD mice.

In [ ]:
rows = []
for z in ZS + ZV:
    for w in W2:
        for D in ["D5", "D15"]:
            col = f"{z}|{w}|{D}"
            mt = c2.groupby(["diet", "subject"])[col].mean().reset_index()
            n, h = mt[mt.diet == "NCD"][col], mt[mt.diet == "HFD"][col]
            t = stats.ttest_ind(n, h)
            rows.append({"Z": z, "W": w, "D": D, "NCD": n.mean(), "NCD sem": n.sem(), "HFD": h.mean(), "HFD sem": h.sem(),
                         "mice": f"{len(n)}/{len(h)}", "t": t.statistic, "p": t.pvalue,
                         "in tol": f"{int(abs(n.mean() - 0.56) <= 0.10)}{int(abs(h.mean() - 0.28) <= 0.09)}",
                         "NCD>HFD p<0.05": (n.mean() > h.mean()) and t.pvalue < 0.05})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** Under Z2 (1-10 s, D15) the per-mouse delta Z-score is 0.61 +- 0.11 for
NCD and 0.20 +- 0.10 for HFD mice (7 and 6 mice), both within tolerance, with p 0.020. It is the
only Z1 to Z2p combination that puts both in; every one gives NCD above HFD with p < 0.05 (0.008 to
0.030).

### fig2e-lick-vigor

> Throughout the 90-trial meal NCD mice lick at a higher rate during the cue than HFD mice; NCD
> licking begins to decline after about 50 trials and HFD licking after about 20.

In [ ]:
def curve_by_mouse(df, key, n=90):
    """Mean over mice of each mouse's mean over recordings of the per-trial-index value
    (triggered trials only)."""
    M = pvh.per_trial_index(recs, df.path.tolist(), key, n)
    per_mouse = pd.DataFrame(M).groupby(df.subject.to_numpy()).mean()
    return per_mouse.mean(axis=0).to_numpy(), per_mouse

def onset(curve, k=15):
    s = pd.Series(curve).rolling(k, center=True, min_periods=1).mean().to_numpy()
    i_max = int(np.nanargmax(s)) + 1
    base = np.nanmean(curve[:k])
    below = s < base
    j = None
    for i in range(len(s)):
        rest = s[i:][np.isfinite(s[i:])]
        if len(rest) and (rest < base).all():
            j = i + 1
            break
    return i_max, j, s

cn, pm_n = curve_by_mouse(c2[c2.diet == "NCD"], ("lick", "cue"))
ch, pm_h = curve_by_mouse(c2[c2.diet == "HFD"], ("lick", "cue"))
both = np.isfinite(cn) & np.isfinite(ch)
print("Trial indices with data in both groups:", int(both.sum()), "of 90")
print("(a) NCD above HFD at every such index:", bool((cn[both] > ch[both]).all()),
      f"; indices where NCD <= HFD: {(np.where(both & (cn <= ch))[0] + 1).tolist()}")
blocks = [(i, i + 15) for i in range(0, 90, 15)]
bl = [(f"{a + 1}-{b}", np.nanmean(cn[a:b]), np.nanmean(ch[a:b])) for a, b in blocks]
print("(b) 15-trial blocks (NCD, HFD):", [(l, round(x, 2), round(y, 2)) for l, x, y in bl],
      "; NCD above in every block:", all(x > y for _, x, y in bl))
for name, c in [("NCD", cn), ("HFD", ch)]:
    i_max, j, s = onset(c)
    print(f"{name}: (i) index of the maximum of the 15-trial moving mean = {i_max}; "
          f"(ii) first index after which the moving mean stays below the mean of trials 1-15 = {j}; "
          f"mean of trials 1-15 = {np.nanmean(c[:15]):.2f} licks/s")
print("Expected onsets: NCD 50 +- 5, HFD 20 +- 2.")
print("\nMice contributing per trial index (NCD, HFD) at 1, 30, 60, 90:",
      [(i, int(pm_n[i - 1].notna().sum()), int(pm_h[i - 1].notna().sum())) for i in [1, 30, 60, 90]])
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(np.arange(1, 91), cn, label="NCD"); ax.plot(np.arange(1, 91), ch, label="HFD")
ax.set_xlabel("trial"); ax.set_ylabel("cue lick rate (licks/s)"); ax.legend()
print(f"Plotted ranges: NCD {np.nanmin(cn):.2f} to {np.nanmax(cn):.2f}, HFD {np.nanmin(ch):.2f} to {np.nanmax(ch):.2f}")
plt.tight_layout(); plt.show()

**Result: partially reproduced.** NCD mice lick more during the cue than HFD mice at all 90 trial
indices and in every 15-trial block (4.36 to 5.12 against 2.59 to 4.07 licks/s). The onset of the
decline is later for NCD than for HFD under both readings but out of tolerance: (i) the maximum of
the 15-trial moving mean is at trial 22 for NCD and 13 for HFD; (ii) the moving mean stays below
its first-15 mean from trial 57 for NCD and 27 for HFD, against 50 +- 5 and 20 +- 2.

### fig2fj-early-lick-by-meal-size

> Sessions with smaller meals have lower lick rates early in the meal; initial cue lick rates span
> 4 to 5.5 licks/s across NCD meal-size groups and 2 to 5.7 licks/s across HFD groups.

In [ ]:
rows = []
for reading, col in [("(a) first triggered trial", "lick_cue|first"), ("(b) first 15 triggered", "lick_cue|early")]:
    g = c2_15.groupby(["diet", "meal_bin"])[col].agg(["mean", "sem", "count"])
    print(reading); print(g.round(2).to_string())
    res = {"reading": reading}
    for d, (lo_e, hi_e) in {"NCD": (4.0, 5.5), "HFD": (2.0, 5.7)}.items():
        m = g.loc[d]["mean"]
        res[f"{d} lowest"] = m.min(); res[f"{d} highest"] = m.max()
        res[f"{d} in tol"] = (abs(m.min() - lo_e) <= 0.1 * lo_e) and (abs(m.max() - hi_e) <= 0.1 * hi_e)
        res[f"{d} smallest<largest group"] = m.iloc[0] < m.iloc[-1]
    rows.append(res)
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.2f}"))

**Result: reproduced.** With the initial lick rate read as the mean of the first 15 triggered
trials, the NCD meal-size groups span 3.91 to 5.38 licks/s and the HFD groups 1.89 to 5.76, all
four bounds within tolerance of 4 to 5.5 and 2 to 5.7, and in each diet the smallest meal-size
group has a lower mean than the largest. With the first triggered trial alone the bounds are
lower (NCD 3.00 to 4.43, HFD 1.33 to 3.00).

### fig2p-lick-rate-meal-size

> The mid-meal cue lick rate correlates positively with meal size in both NCD and HFD sessions.

In [ ]:
rows = []
for reading, col in [("mid-meal, 15th-29th", "lick_cue|mid"), ("early, first 15", "lick_cue|early")]:
    for d in ["NCD", "HFD"]:
        x = c2_15[c2_15.diet == d]
        r, p, n = pearson(x.n_triggered.to_numpy(float), x[col].to_numpy(float))
        rows.append({"reading": reading, "diet": d, "r": r, "p": p, "n": n})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** The mid-meal (15th to 29th triggered trial) cue lick rate correlates
with meal size in NCD (r 0.44, p 0.0006) and HFD sessions (r 0.52, p 0.0001); the early-meal
reading gives r 0.48 and 0.61.

### fig2km-elevated-early-at-matched-licking

> In sessions of 45 to 74 triggered trials, where early licking vigor is comparable between
> groups, HFD mice still have larger PVH MC4R responses early in the meal than NCD mice.

In [ ]:
rows = []
for z in ZS + ZV:
    for w in W2:
        for b in ["45-59", "60-74"]:
            x = c2_15[c2_15.meal_bin == b]
            col = f"{z}|{w}|early"
            n, h = x[x.diet == "NCD"][col].dropna(), x[x.diet == "HFD"][col].dropna()
            ln, lh = x[x.diet == "NCD"]["lick_cue|early"], x[x.diet == "HFD"]["lick_cue|early"]
            rows.append({"Z": z, "W": w, "group": b, "n": f"{len(n)}/{len(h)}", "NCD resp": n.mean(), "HFD resp": h.mean(),
                         "diff": h.mean() - n.mean(), "sum sem": n.sem() + h.sem(),
                         "resp HFD>NCD by > sum sem": h.mean() - n.mean() > n.sem() + h.sem(),
                         "NCD lick": ln.mean(), "HFD lick": lh.mean(), "lick |diff|": abs(lh.mean() - ln.mean()),
                         "lick sum sem": ln.sem() + lh.sem(),
                         "lick comparable": abs(lh.mean() - ln.mean()) < ln.sem() + lh.sem()})
F2KM = pd.DataFrame(rows)
F2KM["both"] = F2KM["resp HFD>NCD by > sum sem"] & F2KM["lick comparable"]
print(F2KM.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** In the 45-59 and 60-74 meal-size groups the early-meal cue lick rates
differ by 0.12 and 0.02 licks/s, less than the sums of their s.e.m. (0.86 and 0.63), while under
Z2, Z2p and Z0 the HFD early response exceeds the NCD one by 0.37 to 0.77, more than the sums of
the s.e.m. (0.21 to 0.28). Under Z1 and Z1p it holds in the 45-59 group only (in neither for Z1,
1-10 s), and HFD is below NCD in the 60-74 group.

### fig2q-midmeal-response-meal-size

> In HFD sessions, mid-meal PVH MC4R responses correlate negatively with meal size (r = -0.46, p =
> 0.001).

In [ ]:
rows = []
for k in [15, 30]:
    x0 = sel_c2(k)
    for z in ZS + ZV:
        for w in W2:
            for d in ["HFD", "NCD"]:
                x = x0[x0.diet == d]
                r, p, n = pearson(x.n_triggered.to_numpy(float), x[f"{z}|{w}|mid"].to_numpy(float))
                rows.append({"min triggered": k, "Z": z, "W": w, "diet": d, "r": r, "p": p, "n": n,
                             "in tol (HFD)": (abs(r + 0.46) <= 0.046) if d == "HFD" else None})
F2Q = pd.DataFrame(rows)
print(F2Q.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** In HFD recordings with at least 15 triggered trials the mid-meal
response correlates with meal size with r -0.425 (Z2, 1-9 s, p 0.0023) and -0.459 (Z2, 1-10 s, p
0.0009), within -0.46 +- 0.046; Z2p gives -0.47 and -0.50. Under Z1 and Z1p r is -0.27 to -0.29 (p
0.046 to 0.065) in the same recordings. The NCD correlations are -0.15 to -0.18 (p 0.18 to 0.26).

### fig2r-weight-midmeal-response

> In HFD mice, absolute body weight on the recording day correlates positively with mid-meal
> Z-scored responses.

In [ ]:
rows = []
for k in [15, 30]:
    x0 = sel_c2(k)
    for z in ZS + ZV:
        for w in W2:
            for d in ["HFD", "NCD"]:
                x = x0[x0.diet == d]
                r, p, n = pearson(x.bw.to_numpy(float), x[f"{z}|{w}|mid"].to_numpy(float))
                rows.append({"min triggered": k, "Z": z, "W": w, "diet": d, "r": r, "p": p, "n": n})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nHFD body weight per mouse (g):", sel_c2(15)[sel_c2(15).diet == "HFD"].groupby("subject").bw.agg(["mean", "min", "max"]).round(1).to_dict("index"))

**Result: reproduced.** In HFD recordings, body weight correlates positively with the mid-meal
response under every Z1, Z2, Z1p and Z2p reading (r 0.45 to 0.56, p 5e-05 to 0.0014). In NCD
recordings r is -0.02 to -0.21 (p 0.12 to 0.88).

### fig2s-weight-meal-length

> In HFD mice, meal length (triggered trials) correlates negatively with absolute body weight.

In [ ]:
rows = []
for k in [15, 30]:
    x0 = sel_c2(k)
    for d in ["HFD", "NCD"]:
        x = x0[x0.diet == d]
        r, p, n = pearson(x.bw.to_numpy(float), x.n_triggered.to_numpy(float))
        rows.append({"min triggered": k, "diet": d, "r": r, "p": p, "n": n})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** In HFD recordings with at least 15 triggered trials, meal size
correlates negatively with body weight (r -0.38, p 0.006); with at least 30, r -0.27 (p 0.066).

### supp2a-volume-trials

> Ensure volume consumed correlates robustly with the number of triggered trials in both diet
> groups, so meal size is defined as triggered trials.

In [ ]:
x0 = sel_c2(0)
print("Cohort 2 90-trial recordings with a recorded volume:", int(x0.ensure.notna().sum()), "of", len(x0))
rows = []
for d in ["NCD", "HFD"]:
    x = x0[x0.diet == d]
    r, p, n = pearson(x.n_triggered.to_numpy(float), x.ensure.to_numpy(float))
    rows.append({"diet": d, "r": r, "p": p, "n": n, "volume range (mL)": (x.ensure.min(), x.ensure.max())})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** The recorded Ensure volume correlates with the number of triggered
trials in NCD (r 0.68, p 8e-09) and HFD recordings (r 0.79, p 6e-12).

### supp2h-delta-z-meal-size

> Across sessions, the within-meal change in response correlates with meal size with r = -0.34 (p
> = 0.012) in NCD mice and r = 0.26 (p = 0.048) in HFD mice. The sentence calls the NCD
> correlation positive and the HFD one negative, opposite to the printed signs.

In [ ]:
rows = []
for k in [30, 15]:
    x0 = sel_c2(k)
    for z in ZS + ZV:
        for w in W2:
            for D in ["D15", "D5"]:
                res = {"min triggered": k, "Z": z, "W": w, "D": D}
                for d in ["NCD", "HFD"]:
                    x = x0[x0.diet == d]
                    r, p, n = pearson(x.n_triggered.to_numpy(float), x[f"{z}|{w}|{D}"].to_numpy(float))
                    res[f"r {d}"], res[f"p {d}"], res[f"n {d}"] = r, p, n
                res["numbers match"] = abs(res["r NCD"] + 0.34) <= 0.034 and abs(res["r HFD"] - 0.26) <= 0.026 and res["p NCD"] < 0.05 and res["p HFD"] < 0.05
                res["words match"] = res["r NCD"] > 0 and res["r HFD"] < 0 and res["p NCD"] < 0.05 and res["p HFD"] < 0.05
                rows.append(res)
S2H = pd.DataFrame(rows)
print(S2H.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: not reproduced.** No combination gives r near -0.34 (NCD) and 0.26 (HFD). The NCD
correlations are positive (0.13 to 0.22, p 0.11 to 0.33) and the HFD ones negative (-0.003 to
-0.33), the signs of the sentence's words, but the two are never both significant: the HFD
correlation reaches p < 0.05 only under Z2p with D5 (r -0.31 and -0.33, p 0.035 and 0.027), where
the NCD one has p 0.14 and 0.16.

### supp2fg-locomotion-not-meal-size

> HFD mice show reduced locomotion, but locomotion does not correlate with meal size.

In [ ]:
rows = []
for k in [15, 30]:
    x0 = sel_c2(k)
    for key, label in [("run_pre|all", "pre-cue speed (main)"), ("run_rew|all", "after reward 1-10 s (variant)"),
                       ("runsigned_pre|all", "pre-cue signed (variant)")]:
        res = {"min triggered": k, "measure": label,
               "NCD mean": x0[x0.diet == "NCD"][key].mean(), "HFD mean": x0[x0.diet == "HFD"][key].mean()}
        for d in ["NCD", "HFD"]:
            x = x0[x0.diet == d]
            r, p, n = pearson(x.n_triggered.to_numpy(float), x[key].to_numpy(float))
            res[f"r {d}"], res[f"p {d}"], res[f"n {d}"] = r, p, n
        rows.append(res)
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nPer mouse pre-cue running (cm/s), >= 15 triggered:")
print(sel_c2(15).groupby(["diet", "subject"])["run_pre|all"].mean().round(3).to_string())

**Result: reproduced.** HFD pre-cue speed is below NCD (2.13 against 3.98 cm/s, recordings with
at least 15 triggered trials) and the Pearson correlation of meal size with pre-cue speed has p
0.050 (NCD, r 0.26) and 0.19 (HFD, r -0.19); with at least 30 triggered trials p is 0.050 and 0.053.
The after-reward variant has HFD p 0.018 with at least 30 triggered trials.

### supp2j-variability-weight

> Pre-Ensure spontaneous signal variability correlates weakly with absolute body weight in HFD
> mice.

In [ ]:
rows = []
for k in [0, 30]:
    x0 = sel_c2(k)
    for w in pvh.ROLL_S:
        for d in ["HFD", "NCD"]:
            x = x0[x0.diet == d]
            r, p, n = pearson(x.bw.to_numpy(float), x[f"rollsd_{w}s"].to_numpy(float))
            rows.append({"min triggered": k, "rolling window (s)": w, "diet": d, "r": r, "p": p, "n": n,
                         "weak & p<0.05": (abs(r) < 0.5) and p < 0.05})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
pre = [r["pre_ensure_s"] for r in recs if "pre_ensure_s" in r]
print(f"\nPre-Ensure period length (s): min {min(pre):.0f}, median {np.median(pre):.0f}, max {max(pre):.0f}")

**Result: reproduced.** In HFD recordings the rolling SD of dF/F0 before the first Ensure
correlates with body weight under all three windows, weakly (r 0.34 to 0.42, p 0.003 to 0.018). The
NCD correlation is significant with the 1-s window only (r 0.35).

### supp2k-sex-early-response

> NCD females show slightly larger early-meal responses than males, a difference more pronounced in
> HFD mice.

In [ ]:
print("Recordings and mice by sex and diet (legend: M-NCD 26/3, M-HFD 31/4, F-NCD 33/4, F-HFD 23/3):")
print(c2.groupby(["sex", "diet"]).agg(recordings=("path", "size"), mice=("subject", "nunique")).to_string())
rows = []
for z in ZS + ZV:
    for w in W2:
        g = c2.groupby(["diet", "sex"])[f"{z}|{w}|early"].mean()
        dn, dh = g[("NCD", "F")] - g[("NCD", "M")], g[("HFD", "F")] - g[("HFD", "M")]
        rows.append({"Z": z, "W": w, "M-NCD": g[("NCD", "M")], "F-NCD": g[("NCD", "F")], "M-HFD": g[("HFD", "M")],
                     "F-HFD": g[("HFD", "F")], "F-M NCD": dn, "F-M HFD": dh, "F>M both": dn > 0 and dh > 0,
                     "larger in HFD": dh > dn})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nPer mouse early response (Z2, 1-9 s):")
print(c2.groupby(["diet", "sex", "subject"])["z2|1-9|early"].agg(["mean", "count"]).round(3).to_string())

**Result: reproduced.** Under every Z1, Z2, Z1p and Z2p combination females have larger
early-meal responses than males in both diets, and the difference is larger in HFD (for Z2, 1-9 s:
0.19 in NCD, 0.46 in HFD). Under Z0 the difference is smaller in HFD (0.13 to 0.18 against 0.30 to
0.33).

### supp2lp-fasting-no-effect

> An additional overnight fast does not significantly alter licking or PVH MC4R response dynamics
> in either diet group.

In [ ]:
print("Recordings by condition (legend: nonfasted NCD 31, fasted NCD 28, nonfasted HFD 31, fasted HFD 23; 7 mice each):")
print(c2.groupby(["diet", "condition"]).agg(recordings=("path", "size"), mice=("subject", "nunique")).to_string())
measures = [("cue lick rate", "lick_cue|all")]
for z in ZS + ZV:
    for w in W2:
        measures += [(f"early {z} {w}", f"{z}|{w}|early"), (f"D15 {z} {w}", f"{z}|{w}|D15")]
rows = []
for d in ["NCD", "HFD"]:
    x = c2[c2.diet == d].copy()
    x["fasted"] = (x.condition == "fasted").astype(int)
    for name, col in measures:
        a, b = x[x.fasted == 1][col].dropna(), x[x.fasted == 0][col].dropna()
        t = stats.ttest_ind(a, b)
        y = x[["fasted", "subject", col]].dropna().rename(columns={col: "v"})
        m = smf.mixedlm("v ~ fasted", y, groups=y.subject).fit(reml=True)
        rows.append({"diet": d, "measure": name, "fasted": a.mean(), "nonfasted": b.mean(), "diff": a.mean() - b.mean(),
                     "p t-test": t.pvalue, "LME coef": m.params["fasted"], "p LME": m.pvalues["fasted"]})
S2L = pd.DataFrame(rows)
print(S2L.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
for zs, lab in [(ZS, "Z1/Z2 readings"), (ZV, "Z0 variant")]:
    keep = S2L.measure.str.contains("lick") | S2L.measure.str.contains("|".join(zs))
    s = S2L[keep]
    print(f"{lab}: tests with p < 0.05 -- t-test: {int((s['p t-test'] < 0.05).sum())} of {len(s)}, "
          f"LME: {int((s['p LME'] < 0.05).sum())} of {len(s)}")

**Result: not reproduced.** Both test readings find an effect of fasting in HFD sessions: the
cue lick rate is higher after fasting (4.01 against 3.29 licks/s; t-test p 0.037, LME p 0.005)
and the early-meal response is lower under Z2 and Z2p (t-test p 0.007 to 0.024, LME p 0.004 to
0.010; under Z1 and Z1p LME p 0.021 to 0.027). In NCD sessions no test has p < 0.05. The delta
Z-score does not differ in either diet.

### supp2q-hfd-more-training

> HFD mice needed more training sessions than NCD mice to reach stable task performance.

In [ ]:
print("Training sessions in the dataset: the session descriptions are",
      sorted({r["description"] for r in recs}), "; every file is a photometry recording with the 60- or 90-trial task.")
print("Earliest recording per cohort-2 mouse:", U[(U.cohort == 2)].groupby("subject").date.min().to_dict())

**Result: blocked (data missing).** The dataset holds the recording sessions only; no file
describes a training session and no metadata counts them.

## Fig. 3 and Supp. Fig. 3: diet switch

First-diet recordings are the cohort recordings of the switched mice, second-diet recordings the
files with `startdiet` and `2nd_diet` in their notes. Recordings with at least 30 triggered
trials (Fig. 3 legend) unless said otherwise. One sigma per mouse over both phases (see `pvh`).

In [ ]:
SW = R[R.switch.notna()].copy()
SW["phase_label"] = np.where(SW.phase == "first", "before", "after")
SW30 = SW[SW.n_triggered >= 30]
THREE = ["NCD->NCD", "HFD->NCD", "HFD->HFD"]
print("Body weight fields in the notes, per mouse and phase (ad lib bw; before ad lib bw):")
print(SW.groupby(["switch", "subject", "phase"])[["adlib_bw", "before_adlib_bw"]].agg(lambda s: sorted(set(s.dropna()))).to_string())

In the second-phase files `ad lib bw` differs from the first-phase value and `before ad lib bw`
is given, so `before ad lib bw` is read as A of Supp. Fig. 3k (weight before the second ad
libitum exposure) and the second-phase `ad lib bw` as B (the maximal ad libitum weight). This
reading of the field names is consistent with the values (B above A for the mice switched to
HFD), not shown by any description in the files.

### fig3h-early-response-diet-switch

> After switching from HFD to NCD, early-meal PVH MC4R responses fall from their level on HFD
> toward NCD levels, while HFD->HFD mice keep high early-meal responses.

In [ ]:
rows = []
for z in ZS + ZV:
    for w in W2:
        x = SW30[SW30.switch.isin(THREE)]
        col = f"{z}|{w}|early"
        long = x[["switch", "phase_label", col]].dropna().rename(columns={col: "Value", "switch": "A", "phase_label": "B"})
        tab, tkd = anova_tukey(long)
        p1, d1 = tukey_p(tkd, "HFD->NCD|after", "HFD->NCD|before")
        p2, d2 = tukey_p(tkd, "HFD->HFD|after", "HFD->HFD|before")
        p3, d3 = tukey_p(tkd, "HFD->NCD|before", "NCD->NCD|before")
        p4, d4 = tukey_p(tkd, "HFD->HFD|before", "NCD->NCD|before")
        g = long.groupby(["A", "B"]).Value.mean()
        rows.append({"Z": z, "W": w, "HFD->NCD before": g[("HFD->NCD", "before")], "HFD->NCD after": g[("HFD->NCD", "after")],
                     "after-before HFD->NCD": d1, "Tukey p": p1,
                     "HFD->HFD before": g[("HFD->HFD", "before")], "HFD->HFD after": g[("HFD->HFD", "after")],
                     "after-before HFD->HFD": d2, "Tukey p HH": p2,
                     "NCD->NCD before": g[("NCD->NCD", "before")], "NCD->NCD after": g[("NCD->NCD", "after")],
                     "HN-NN before": d3, "p HN-NN": p3, "HH-NN before": d4, "p HH-NN": p4})
F3H = pd.DataFrame(rows)
F3H["full"] = (F3H["after-before HFD->NCD"] < 0) & (F3H["Tukey p"] < 0.05) & ~((F3H["after-before HFD->HFD"] < 0) & (F3H["Tukey p HH"] < 0.05))
print(count(SW30[SW30.switch.isin(THREE)], ["switch", "phase_label"]).to_string())
print(F3H.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: partially reproduced.** After the switch the HFD->NCD early-meal response is lower than
before under every reading (Z2 1-9 s: 1.00 to 0.72), but the Tukey test does not find it
significant (p 0.11 to 0.99). HFD->HFD after is not significantly below before (p 0.58 to 0.99).
Before the switch both HFD groups are above NCD->NCD under Z2 and Z2p (p < 0.001); under Z1 and
Z1p only HFD->HFD is.

### fig3i-ncd-increase-only

> After the diet switch, NCD->NCD mice show a significant increase in response from early to
> late in the meal, while HFD->NCD mice show only a slight, nonsignificant increase.

In [ ]:
x2 = SW30[(SW30.phase == "second") & SW30.switch.isin(THREE)]
rows = []
for z in ZS + ZV:
    for w in W2:
        res = {"Z": z, "W": w}
        for ref in THREE:
            m = lme_diet_phase(x2, z, w, group="switch", ref=ref)
            k = [n for n in m.params.index if "Phase" in n and ":" not in n][0]
            res[f"LME late-early {ref}"] = m.params[k]; res[f"p LME {ref}"] = m.pvalues[k]
        mt = mouse_table(x2, [f"{z}|{w}|early", f"{z}|{w}|late"], by="switch")
        for g in THREE:
            y = mt[mt.switch == g]
            t = stats.ttest_rel(y[f"{z}|{w}|late"], y[f"{z}|{w}|early"])
            res[f"paired diff {g}"] = (y[f"{z}|{w}|late"] - y[f"{z}|{w}|early"]).mean(); res[f"p paired {g}"] = t.pvalue
        rows.append(res)
F3I = pd.DataFrame(rows)
F3I["NN increase (both p<0.05)"] = (F3I["LME late-early NCD->NCD"] > 0) & (F3I["p LME NCD->NCD"] < 0.05) & (F3I["paired diff NCD->NCD"] > 0) & (F3I["p paired NCD->NCD"] < 0.05)
F3I["HN absence (neither p<0.05, late>early)"] = (F3I["p LME HFD->NCD"] >= 0.05) & (F3I["p paired HFD->NCD"] >= 0.05) & (F3I["paired diff HFD->NCD"] > 0)
F3I["HN both tests p<0.05"] = (F3I["p LME HFD->NCD"] < 0.05) & (F3I["p paired HFD->NCD"] < 0.05)
print(count(x2, "switch").to_string())
print(F3I.T.to_string(float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** Under Z2 and Z2p (both windows), NCD->NCD mice increase from early to late
after the switch in the LME (p 0.002 to 0.003) and the paired t-test (p 0.004 to 0.006), and
HFD->NCD mice show a small increase that neither test finds significant (LME p 0.40 to 0.60,
paired p 0.14 to 0.35). Under Z1 and Z1p the paired test does not find the NCD->NCD increase (p
0.32 to 0.39) and HFD->NCD late is below early.

### fig3j-delta-z-diet-switch

> After the switch, the per-mouse early-to-late change in response is 0.37 +- 0.03 in NCD->NCD,
> 0.14 +- 0.05 in HFD->NCD and -0.05 +- 0.07 in HFD->HFD mice; it is significantly smaller in
> HFD->HFD.

In [ ]:
EXP_3J = {"NCD->NCD": (0.37, 0.03), "HFD->NCD": (0.14, 0.05), "HFD->HFD": (-0.05, 0.07)}
rows = []
for z in ZS + ZV:
    for w in W2:
        for D in ["D5", "D15"]:
            col = f"{z}|{w}|{D}"
            mt = x2.groupby(["switch", "subject"])[col].mean().reset_index().dropna()
            groups = [mt[mt.switch == g][col] for g in THREE]
            an = stats.f_oneway(*groups)
            tkd = tukey(mt[col], mt.switch)
            p, d = tukey_p(tkd, "HFD->HFD", "NCD->NCD")
            res = {"Z": z, "W": w, "D": D}
            inside = []
            for g in THREE:
                v = mt[mt.switch == g][col]
                res[g] = v.mean(); res[f"{g} sem"] = v.sem()
                inside.append(abs(v.mean() - EXP_3J[g][0]) <= EXP_3J[g][1])
            res.update({"ANOVA p": an.pvalue, "HH-NN": d, "Tukey p HH-NN": p, "in tol": "".join(str(int(i)) for i in inside),
                        "order NN>HN>HH": res["NCD->NCD"] > res["HFD->NCD"] > res["HFD->HFD"]})
            res["full"] = all(inside) and d < 0 and p < 0.05
            res["partial"] = res["order NN>HN>HH"] and d < 0 and p < 0.05
            rows.append(res)
F3J = pd.DataFrame(rows)
print("Mice per group:", x2.groupby("switch").subject.nunique().to_dict())
print(F3J.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: not reproduced.** No combination puts the three means within tolerance, and HFD->HFD
is never significantly below NCD->NCD (Tukey p 0.10 to 0.57). Under Z2 with D15 the means are 0.24,
0.06 and 0.05 for NCD->NCD, HFD->NCD and HFD->HFD; the order NN > HN > HH holds under Z1 and some
Z2 combinations.

### fig3k-ensure-after-switch

> After the switch, HFD->NCD mice consume as much Ensure per session as NCD->NCD mice and
> significantly more than HFD->HFD mice.

In [ ]:
rows = []
for label, df in [(">= 30 triggered", SW30), ("all recordings", SW)]:
    for vol, col in [("recorded", "ensure"), ("triggered x 50 uL", "n_triggered")]:
        x = df[df.switch.isin(THREE)].copy()
        x["Value"] = x[col] * (0.05 if col == "n_triggered" else 1)
        long = x[["switch", "phase_label", "Value"]].dropna().rename(columns={"switch": "A", "phase_label": "B"})
        tab, tkd = anova_tukey(long)
        p1, d1 = tukey_p(tkd, "HFD->NCD|after", "HFD->HFD|after")
        p2, d2 = tukey_p(tkd, "HFD->NCD|after", "NCD->NCD|after")
        g = long.groupby(["A", "B"]).Value.mean()
        rows.append({"recordings": label, "volume": vol, "NN after": g[("NCD->NCD", "after")], "HN after": g[("HFD->NCD", "after")],
                     "HH after": g[("HFD->HFD", "after")], "HN-HH": d1, "Tukey p HN-HH": p1, "HN-NN": d2, "Tukey p HN-NN": p2,
                     "HN before": g[("HFD->NCD", "before")], "full": d1 > 0 and p1 < 0.05 and p2 >= 0.05})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** With all recordings and the recorded volume, HFD->NCD mice consume 2.27
mL after the switch against 1.68 for HFD->HFD (Tukey p 0.007) and 2.37 for NCD->NCD (p 0.99). With
the recordings of at least 30 triggered trials the first comparison is not significant (2.35
against 1.90, p 0.12).

### fig3-hfd-hfd-short-meals

> In the second diet phase, HFD->HFD mice stop consuming Ensure early, never exceeding 60
> triggered trials.

In [ ]:
hh = SW[(SW.phase == "second") & (SW.switch == "HFD->HFD")]
for label, x in [("all second-phase HFD->HFD recordings", hh), ("90-trial sessions only", hh[hh.n_trials >= 89])]:
    print(f"{label}: n = {len(x)}, max triggered = {x.n_triggered.max()}, sessions above 60 = {int((x.n_triggered > 60).sum())}")
print(hh.groupby("subject").n_triggered.agg(lambda s: sorted(s)).to_string())
print("\nTrials per session:", hh.n_trials.value_counts().to_dict())

**Result: not reproduced.** Two of the 30 second-phase HFD->HFD recordings exceed 60 triggered
trials (66 for MPK182 and 64 for MPK187), as do 2 of the 17 90-trial sessions; the maximum is 66.

### fig3c-hfd-ncd-licking-recovers

> After at least six weeks on NCD, formerly HFD mice lick with more vigor during the cue, a
> reinstated motivation to consume food.

In [ ]:
hn = SW30[SW30.switch == "HFD->NCD"]
res = {}
curves = {}
for ph in ["first", "second"]:
    x = hn[hn.phase == ph]
    M = pvh.per_trial_index(recs, x.path.tolist(), ("lick", "cue"), 90)
    curves[ph] = np.nanmean(M, axis=0)
    res[ph] = x["lick_cue|all"]
both = np.isfinite(curves["first"]) & np.isfinite(curves["second"])
print("Recordings (before, after):", len(res["first"]), len(res["second"]))
print(f"Curve mean over trial indices with data in both phases ({int(both.sum())} indices): "
      f"before {np.nanmean(curves['first'][both]):.2f}, after {np.nanmean(curves['second'][both]):.2f} licks/s")
print(f"Mean over recordings of each recording's mean cue lick rate: before {res['first'].mean():.2f} (sem {res['first'].sem():.2f}), "
      f"after {res['second'].mean():.2f} (sem {res['second'].sem():.2f})")
print("Per mouse mean cue lick rate (before, after):")
print(hn.groupby(["subject", "phase"])["lick_cue|all"].mean().unstack().round(2).to_string())
print("\nThe first phase mixes trial counts:", hn.groupby("phase").n_trials.value_counts().to_dict())

**Result: reproduced.** In HFD->NCD mice the cue lick-rate curve averages 4.96 licks/s after the
switch against 3.38 before, over the 90 trial indices, and the mean of the recording means is 5.07
against 3.64; all 7 mice lick more after.

### supp3c-ncd-hfd6w-dynamics-retained

> NCD mice switched to HFD for 6 weeks keep their response dynamics, with a significant
> early-to-late increase in response after the switch, comparable to the change before it, and
> stable licking vigor.

In [ ]:
nh = SW30[SW30.switch == "NCD->HFD6w"]
rows = []
for z in ZS + ZV:
    for w in ["1-9", "1-19"]:
        res = {"Z": z, "W": w}
        for ph, lab in [("first", "NCD"), ("second", "HFD6w")]:
            mt = mouse_table(nh[nh.phase == ph], [f"{z}|{w}|early", f"{z}|{w}|late"], by="switch")
            t = stats.ttest_rel(mt[f"{z}|{w}|late"], mt[f"{z}|{w}|early"])
            res[f"{lab} early"] = mt[f"{z}|{w}|early"].mean(); res[f"{lab} late"] = mt[f"{z}|{w}|late"].mean()
            res[f"{lab} p"] = t.pvalue; res[f"{lab} n mice"] = len(mt)
        res["both phases late>early p<0.05"] = (res["NCD late"] > res["NCD early"]) and res["NCD p"] < 0.05 and (res["HFD6w late"] > res["HFD6w early"]) and res["HFD6w p"] < 0.05
        res["HFD6w only"] = (res["HFD6w late"] > res["HFD6w early"]) and res["HFD6w p"] < 0.05
        rows.append(res)
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nMean cue lick rate per phase (recordings):", nh.groupby("phase")["lick_cue|all"].agg(["mean", "sem", "count"]).round(2).to_dict("index"))

**Result: reproduced.** Under Z2 and Z2p in the 1-9 s window, NCD->HFD6w mice increase from
early to late both before the switch (p 0.0002) and after it (p 0.011). Under Z1 and Z1p the
increase after the switch is not significant (p 0.06 to 0.09). The mean cue lick rate is 4.26
before and 4.43 licks/s after.

### supp3j-body-weight-switch

> NCD->HFD6w mice gain significant body weight over the 6-week HFD phase, while HFD->NCD mice do
> not lose weight (a slight, nonsignificant gain).

In [ ]:
rows = []
for label, df in [(">= 30 triggered", SW30), ("all recordings", SW)]:
    for g in ["NCD->NCD", "NCD->HFD6w", "HFD->NCD", "HFD->HFD"]:
        x = df[df.switch == g].copy()
        x["after"] = (x.phase == "second").astype(int)
        m = smf.mixedlm("bw ~ after", x, groups=x.subject).fit(reml=True)
        rows.append({"recordings": label, "condition": g, "before g": x[x.after == 0].bw.mean(), "after g": x[x.after == 1].bw.mean(),
                     "phase coef": m.params["after"], "p": m.pvalues["after"], "n": len(x), "mice": x.subject.nunique()})
S3J = pd.DataFrame(rows)
print(S3J.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** NCD->HFD6w mice gain 6.5 g (LME p 6e-34) and HFD->NCD mice gain a
nonsignificant 0.5 g (p 0.15; with all recordings 0.6 g, p 0.09).

### supp3m-weight-gain-response

> In the second diet phase, mid-meal responses correlate positively with weight gain in
> NCD->HFD6w mice (r = 0.53, p = 0.001) and also in NCD->NCD mice (r = 0.46, p = 0.003).

In [ ]:
rows = []
for k in [30, 15]:
    x0 = SW[(SW.phase == "second") & (SW.n_triggered >= k)].copy()
    x0["C-A"] = x0.bw - x0.before_adlib_bw
    x0["C/A"] = x0.bw / x0.before_adlib_bw
    for z in ZS + ZV:
        for w in W2:
            for gain in ["C-A", "C/A"]:
                res = {"min triggered": k, "Z": z, "W": w, "gain": gain}
                for g, (rr, tol) in {"NCD->HFD6w": (0.53, 0.053), "NCD->NCD": (0.46, 0.046)}.items():
                    x = x0[x0.switch == g]
                    r, p, n = pearson(x[gain].to_numpy(float), x[f"{z}|{w}|mid"].to_numpy(float))
                    res[f"r {g}"], res[f"p {g}"], res[f"n {g}"] = r, p, n
                    res[f"tol {g}"] = abs(r - rr) <= tol and p < 0.05
                rows.append(res)
S3M = pd.DataFrame(rows)
print(S3M.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: partially reproduced.** In recordings with at least 30 triggered trials, the reading
adopted, the mid-meal response correlates positively with the weight change in NCD->HFD6w (r
0.28 to 0.65) and NCD->NCD mice (r 0.29 to 0.34), p < 0.05 in both groups under most
combinations, but no combination puts both r within tolerance (NCD->NCD at most 0.34 against 0.46
+- 0.046). With at least 15 triggered trials, Z2p (1-10 s, C/A) puts both in (0.57 and 0.42, p
0.0001 and 0.004).

### supp3l-weight-not-meal-length

> In the second diet phase, absolute body weight does not predict meal length in NCD->NCD or
> NCD->HFD6w mice.

In [ ]:
rows = []
for label, k in [(">= 30 triggered (Methods default; decides)", 30), ("all recordings (variant)", 0)]:
    x0 = SW[(SW.phase == "second") & (SW.n_triggered >= k)]
    for g in ["NCD->NCD", "NCD->HFD6w"]:
        x = x0[x0.switch == g]
        r, p, n = pearson(x.bw.to_numpy(float), x.n_triggered.to_numpy(float))
        rows.append({"recordings": label, "group": g, "r": r, "p": p, "n": n})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** In the second phase, body weight does not correlate with meal size in
NCD->NCD (r -0.05, p 0.75) or NCD->HFD6w mice (r -0.14, p 0.42); with all recordings p is 0.75 and
0.49.

### supp3q-hfd-ncd-weight-loss-meal-length

> In HFD->NCD mice, meal length correlates negatively with weight change, so mice that lost more
> weight before the session tended to eat larger meals; neural responses do not correlate with
> weight change.

In [ ]:
x0 = SW30[(SW30.phase == "second") & (SW30.switch == "HFD->NCD")].copy()
readings = {"C-B (max ad lib)": x0.bw - x0.adlib_bw, "C/B (max ad lib)": x0.bw / x0.adlib_bw,
            "C-A (start of ad lib, legend)": x0.bw - x0.before_adlib_bw, "C/A (start of ad lib, legend)": x0.bw / x0.before_adlib_bw}
print("Reading (b) of the text, weight before HFD exposure (5 weeks of age), is not in the files.")
rows = []
for name, wc in readings.items():
    r1, p1, n1 = pearson(wc.to_numpy(float), x0.n_triggered.to_numpy(float))
    res = {"weight change": name, "r meal length": r1, "p meal length": p1, "n": n1}
    worst_i, worst_ii = 1.0, 1.0
    for z in ZS:
        for w in W2:
            ri, pi, _ = pearson(wc.to_numpy(float), x0[f"{z}|{w}|mid"].to_numpy(float))
            rii, pii, _ = pearson(x0.n_triggered.to_numpy(float), x0[f"{z}|{w}|mid"].to_numpy(float))
            res[f"r(i) {z} {w}"], res[f"p(i) {z} {w}"] = ri, pi
            res[f"r(ii) {z} {w}"], res[f"p(ii) {z} {w}"] = rii, pii
            worst_i, worst_ii = min(worst_i, pi), min(worst_ii, pii)
    res["min p (i)"], res["min p (ii)"] = worst_i, worst_ii
    rows.append(res)
S3Q = pd.DataFrame(rows)
print(S3Q.T.to_string(float_format=lambda v: f"{v:.3g}"))
print("\nVariant Z0:")
for name, wc in readings.items():
    ri, pi, _ = pearson(wc.to_numpy(float), x0["z0|1-9|mid"].to_numpy(float))
    rii, pii, _ = pearson(x0.n_triggered.to_numpy(float), x0["z0|1-9|mid"].to_numpy(float))
    print(f"  {name}: (i) r={ri:.3f} p={pi:.3g}; (ii) r={rii:.3f} p={pii:.3g}")

**Result: reproduced.** In HFD->NCD second-phase recordings, meal size correlates negatively with
the weight on the day relative to the second-phase ad libitum weight, as a ratio (r -0.35, p
0.023); as a difference, and relative to the weight before the second ad libitum phase, r is -0.24
to -0.27 (p 0.09 to 0.14). The mid-meal response correlates neither with the weight change (p
0.40 to 0.998) nor, under Z2 and Z2p, with meal size (p 0.14 to 0.35); under Z1 it does with meal
size (r -0.32, p 0.04).

## Summary

| Claim | Outcome |
|---|---|
| fig1k-early-late-recordings | Partially reproduced |
| fig1k-lme | Not reproduced |
| fig1l-early-late-mice | Partially reproduced |
| supp1i-hfd-no-change-1-20s | Reproduced |
| supp1h-dff-early-late | Reproduced |
| fig1gh-hfd-less-ensure | Reproduced |
| fig1c-hfd-heavier | Partially reproduced |
| supp1cd-licking-declines | Blocked: methods underspecified |
| supp1e-hfd-less-locomotion | Reproduced |
| fig2b-meal-size | Reproduced |
| fig2c-delta-z-recordings | Reproduced |
| fig2d-delta-z-mice | Reproduced |
| fig2e-lick-vigor | Partially reproduced |
| fig2fj-early-lick-by-meal-size | Reproduced |
| fig2p-lick-rate-meal-size | Reproduced |
| fig2km-elevated-early-at-matched-licking | Reproduced |
| fig2q-midmeal-response-meal-size | Reproduced |
| fig2r-weight-midmeal-response | Reproduced |
| fig2s-weight-meal-length | Reproduced |
| supp2a-volume-trials | Reproduced |
| supp2h-delta-z-meal-size | Not reproduced |
| supp2fg-locomotion-not-meal-size | Reproduced |
| supp2j-variability-weight | Reproduced |
| supp2k-sex-early-response | Reproduced |
| supp2lp-fasting-no-effect | Not reproduced |
| supp2q-hfd-more-training | Blocked: data missing |
| fig3h-early-response-diet-switch | Partially reproduced |
| fig3i-ncd-increase-only | Reproduced |
| fig3j-delta-z-diet-switch | Not reproduced |
| fig3k-ensure-after-switch | Reproduced |
| fig3-hfd-hfd-short-meals | Not reproduced |
| fig3c-hfd-ncd-licking-recovers | Reproduced |
| supp3c-ncd-hfd6w-dynamics-retained | Reproduced |
| supp3j-body-weight-switch | Reproduced |
| supp3m-weight-gain-response | Partially reproduced (assumed) |
| supp3l-weight-not-meal-length | Reproduced |
| supp3q-hfd-ncd-weight-loss-meal-length | Reproduced (assumed) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The Z-score of the paper could not be rebuilt
exactly: the claims file gives two readings of the Methods' per-animal Z-score (Z1, Z2), run here
from the deposited raw 465 nm trace, with the per-mouse SD over all recordings or over one diet
phase. The trial onsets in the raw trace are found from the deposited Z-score (or, in 58 files,
taken as the modal onset). Trials with a lost signal (12, in two MPK120 recordings) are left out
of the raw measures. Speed is the absolute value of the signed running signal. For the claims
whose criterion leaves a recording set open, both sets are shown. One cohort-2 HFD mouse of seven,
and a few recordings of others, are not in the dataset (counts above).

What was found about the dataset.

- Four `ses-None` files are byte-identical copies of dated files (MPK118 2024-02-13, MPK133
  2024-02-07, MPK134 2024-02-12, MPK172 2024-07-25); two other `ses-None` files are unique
  sessions with no date in the file name.
- Cohort 2 holds 6 HFD mice, where the paper has 7 (54 recordings); 47 HFD recordings with at
  least 30 triggered trials are in it. Cohort 1 has one recording fewer than the paper per diet
  (38 and 32 against 39 and 33).
- In 32 files the deposited Z-score does not match the raw trace of the same file at any offset.
- The `starting_time` rate is 1 Hz in about half the files, where the data are at 50 Hz.
- In 27 files one series has more trial columns than the Ensure, lick and running matrices.
- The sex of MPK173 and MPK182 differs between their first- and second-phase files; the
  first-phase labels give the paper's sex counts for the female HFD group.
- The age is "P12W/" in every file. One second-phase file of MPK191 is dated 2024-01-02, before
  the mouse's first-phase recordings (October 2024).
- The running signal is signed, with no description of the sign; the body weights, Ensure volume
  and the `ad lib bw` and `before ad lib bw` fields have no units or definitions.
- The 405 nm control, the training sessions and the home-cage food intake are not in the dataset.
